In [ ]:
# Practice Problems Day 7

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 7 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**Reading:** Geron chapter 6, the sections on **Boosting** (AdaBoost and Gradient Boosting) and **Stacking**.

GOALS:

1. Fit a boosted model and compare it to the Day 6 forest on the same tumors.
2. Let XGBoost pick its own number of trees with early stopping, and see what the learning rate does to that number.
3. Use repeated cross validation to decide whether two close models are really different.
4. Find out for yourself whether more trees always overfit.

**How much of this should you write yourself?** Every problem says. Problem 1 gives you the code with blanks. Problems 2 and 3 list the steps. Problem 4 is by hand. Problem 5 just tells you what to do. Same breast cancer data as the Day 5 and Day 6 homework, so the loading and splitting code is code you have written twice.

**The test set stays closed.** You opened it in the Day 5 and Day 6 homework. Everything here happens on validation and cross validation.

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

---

## Problem 1: Boosting on the tumors (fill in the blanks)

**What to copy, what to write:** Fill in the blanks marked `____`. Steps 1 and 2 are the same as the Day 5 and Day 6 homework and are done for you.

**1a.** Fill in the blanks and run it. Report the training accuracy and the validation average precision.

**1b.** In the Day 6 homework, the random forest got a validation average precision of **0.986** on these same tumors. How does your boosted model compare?

**1c.** Look at `max_depth=3` in the code. In the Day 6 forest, every tree was grown all the way down. Why do boosted models use shallow trees instead? (One or two sentences. The Day 7 notes and Geron both talk about this.)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, average_precision_score

# Step 1: load the data, 1 means MALIGNANT (same as the Day 5 and Day 6 homework)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series((data.target == 0).astype(int))

# Step 2: test set off first, then a validation set. The test set stays closed today
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=42)

# Step 3: a boosted model
boosted = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,       # the usual first try from the notes
    max_depth=3,
    random_state=42)
boosted.fit(X_train, y_train)      # the training features and labels. No scaling needed, these are trees

# Step 4: score it
print("training accuracy:", round(accuracy_score(y_train, boosted.predict(X_train)), 3))
y_prob_boost = boosted.predict_proba(X_valid)[:, 1]    # the VALIDATION features, and the column for class 1
print("validation average precision:", round(average_precision_score(y_valid, y_prob_boost), 3))

**1a.** The blanks: `0.1`, `X_train, y_train`, `X_valid` and `1`. Training accuracy 1.0, validation average precision **0.975**.

**1b.** The forest got 0.986 and the boosted model gets 0.975, so the forest is very slightly ahead. Problem 3 is about whether that difference means anything. (It does not.)

**1c.** Each tree in a boosted model only has to fix the leftovers from the trees before it, so it does not need to be a whole model on its own. Shallow trees also add slowly, which is what makes the learning rate work. Deep trees in a boosted model fix the leftovers all at once and overfit fast. A forest is different: no tree corrects another, so each tree is grown all the way and the averaging fixes the overfitting afterward.

---

## Problem 2: Let XGBoost stop itself (steps outlined)

**What to copy, what to write:** The steps are comments in the cell. The early stopping cell in the Day 7 notes is the pattern to follow.

**2a.** Fit an XGBoost model with `n_estimators=2000`, `learning_rate=0.05`, `max_depth=3`, `eval_metric="aucpr"`, and `early_stopping_rounds=50`, watching the validation set. Print `best_iteration` and the validation average precision.

**2b.** Do it again with `learning_rate=0.3` and nothing else changed. Print the same two numbers.

**2c.** Compare the two. What happened to the number of trees, and what happened to the score? Explain it with the learning rate section of the notes.

**2d.** Neither model used anything close to 2000 trees. If you had instead set `n_estimators=2000` with no early stopping, what would have happened, and how would you have found out?

In [ ]:
import xgboost as xgb

for rate in [0.05, 0.3]:
    model = xgb.XGBClassifier(
        n_estimators=2000,
        learning_rate=rate,
        max_depth=3,
        random_state=42,
        eval_metric="aucpr",
        early_stopping_rounds=50)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=False)

    ap = average_precision_score(y_valid, model.predict_proba(X_valid)[:, 1])
    print("learning rate", rate, "  stopped at tree", model.best_iteration,
          "  validation average precision", round(ap, 3))

**2a.** Stops at tree **73**, validation average precision **0.979**.

**2b.** Stops at tree **9**, validation average precision **0.979**.

**2c.** Six times the learning rate, roughly eight times fewer trees, and the same score. Each tree is added at six times the strength, so the model reaches the same place in far fewer steps. This is the trade off from the notes: `learning_rate` and `n_estimators` are one decision. Notice the scores are the same to three decimals, so neither setting is better here.

**2d.** It would have built all 2000 trees, wasting time, and on noisier data it would have started fitting noise after the best point. You would find out the same way the notes did: by scoring the model as trees are added and watching the validation score turn back upward. Early stopping automates exactly that.

---

## Problem 3: Are they really different? (steps outlined)

In 1b you compared 0.975 against the forest's 0.986. Now find out whether that gap is real.

**What to copy, what to write:** The `RepeatedStratifiedKFold` cell in the notes is the pattern. Change the models and the data.

**3a.** Use `cross_val_score` with `RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=42)` and `scoring="average_precision"` on **`X_train_full`** (the full training set, not the split you fit on). Do it for a random forest with 200 trees and for your boosted model. Print each one's mean, standard deviation, and worst and best fold.

**3b.** Which model has the higher mean? By how much? How does that gap compare to the standard deviation?

**3c.** Write the one sentence you would put in a report about which model is better.

**3d.** The test set would settle it in one line. Why are you not allowed to use it here?

In [ ]:
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=42)

models = [("random forest", RandomForestClassifier(n_estimators=200, random_state=42)),
          ("gradient boosting", GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42))]

for name, model in models:
    scores = cross_val_score(model, X_train_full, y_train_full, cv=cv, scoring="average_precision")
    print(name)
    print("   mean:", round(scores.mean(), 3), "  standard deviation:", round(scores.std(), 3))
    print("   worst and best fold:", round(scores.min(), 3), "to", round(scores.max(), 3))

**3a.** Forest: mean **0.985**, standard deviation 0.012, folds from 0.950 to 0.999. Boosting: mean **0.989**, standard deviation 0.006, folds from 0.973 to 1.000.

**3b.** Boosting is ahead by 0.004, and the folds vary by 0.006 to 0.012. The gap is smaller than that variation. Notice it also points the **other way** from 1b, where the forest looked better on the single validation set. That is what noise looks like.

**3c.** Your sentence will be different, here is one: "On repeated cross validation the two models are tied, 0.985 against 0.989, well within the fold-to-fold variation, so there is no evidence that either is better on this data."

A good sentence reports both numbers and does not name a winner.

**3d.** Because using the test set to choose between models is choosing, and then its score is no longer an honest estimate of anything. You already opened it in the Day 5 and Day 6 homework to report a model that had been chosen on validation. Choices are made with validation data or cross validation, always.

---

## Problem 4: By hand, no code

**4a.** A boosted model currently predicts **3.0** for a house, and the true price is **4.2**. What is the residual the next tree will be trained on? If that next tree predicts **1.0** for this house and the learning rate is **0.1**, what does the model predict for the house after the tree is added?

**4b.** You have a boosted model with `learning_rate=0.2` and 100 trees. You want to try `learning_rate=0.02`. Roughly how many trees should you expect to need, and why?

**4c.** Your friend has a 16 core laptop and says "I will just use all the cores, boosting will be as fast as the forest." What is wrong with that plan?

**4d.** Early stopping needs a set of data to watch. Explain why it must not watch the test set, in terms of what the test set is for.

**4a.** Residual 4.2 - 3.0 = **1.2**. After adding the tree: 3.0 + 0.1 x 1.0 = **3.1**. One tree only moves the prediction a little bit, and the learning rate controls how much.

**4b.** About ten times as many, so roughly 1000. Lower the step size by a factor and you need about that factor more steps to cover the same ground.

**4c.** Boosting is sequential: tree 2 cannot start until tree 1 is finished, because it is trained on what tree 1 left over. Extra cores help inside a single tree, but they cannot build the trees at the same time. A forest can use all 16 cores at once because its trees are independent.

**4d.** The test set exists to give one honest number after every choice is made. How many trees to use is a choice, so if the test set decides when to stop, its score includes that choice and is optimistic. Early stopping watches a validation set.

---

## Problem 5: Does it always overfit? (just the task)

In the notes, adding trees past about 50 made the model **worse** on the fake curve: validation error went from 0.362 up to 0.476 by 2000 trees. Find out whether the same thing happens on the tumors.

**5a.** Fit a gradient boosted model with `max_depth=3`, `learning_rate=0.1`, and 10, 50, 200, 1000, and 3000 trees. For each one, print the validation average precision. (The 3000 tree fit takes a few seconds.)

**5b.** Does the score get worse as trees pile up, the way it did on the fake curve?

**5c.** Explain what you found. Two or three sentences. Think about how much noise there is in each dataset: the fake curve had a standard deviation of 0.35 added to every point, while these tumor measurements come from real images of real cells and the two classes barely overlap.

**5d.** Given 5b, does that mean you can always set `n_estimators` as high as you like and stop worrying? Say what you would actually do on a new dataset.

In [ ]:
for n in [10, 50, 200, 1000, 3000]:
    model = GradientBoostingClassifier(n_estimators=n, max_depth=3, learning_rate=0.1, random_state=42)
    model.fit(X_train, y_train)
    ap = average_precision_score(y_valid, model.predict_proba(X_valid)[:, 1])
    print("trees", n, "  validation average precision", round(ap, 3))

**5a.** 10 trees 0.959, 50 trees 0.977, 200 trees 0.975, 1000 trees 0.979, 3000 trees 0.979.

**5b.** No. It rises and then flattens. It never turns back down the way the fake curve did.

**5c.** The fake curve was mostly noise: we added a random amount to every point, so once the boosted model ran out of real pattern, noise was the only thing left to fit. These tumors are nearly separable and the measurements are not very noisy, so the extra trees have almost nothing left to fit and the model just stops changing. Boosting overfits when there is noise in the data for it to fit.

**5d.** No. You cannot tell in advance which kind of data you have, and on a noisy dataset the extra trees cost you real accuracy. The safe habit is the notes' recipe: set `n_estimators` high, turn on early stopping with a validation set, and let it find the number. On clean data it does no harm, and on noisy data it keeps you from overfitting.